# Facial Emotion Recognition using FER2013
**Course:** CST8508 – Machine Vision  
**Student:** Shahin  
**Dataset:** [FER2013 on Kaggle](https://www.kaggle.com/datasets/msambare/fer2013)  

---

## Project Overview

This notebook implements a **Facial Emotion Recognition (FER)** system using deep learning. The system:
- Trains a **MiniXception CNN** on 35,887 grayscale 48×48 facial images
- Classifies faces into **7 emotion categories**: Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise
- Evaluates performance using accuracy curves, confusion matrix, and classification report
- Saves the trained model for deployment in a live webcam demo

### Why MiniXception?
MiniXception (Arriaga et al., 2017) was designed specifically for real-time facial expression recognition. It uses depthwise separable convolutions to achieve strong accuracy with minimal parameters, making it ideal for deployment on consumer hardware.

### Why FER2013?
FER2013 is the standard benchmark dataset for facial emotion recognition, containing 35,887 images collected from Google Image Search. It is deliberately challenging — with noisy labels and high intra-class variation — making it a realistic test of model robustness.

> **Note:** this notebook was run on Google Colab to produce `model/emotion_model.keras`, but the cell outputs
> and training log were not preserved. The architecture cell matches the shipped weights (51,255 parameters);
> `train.py` is the maintained, runnable equivalent. Re-running here will produce a comparable but not identical model.

## Step 1 — Environment Setup

First, we verify the GPU is available and install any required packages. This notebook is designed to run on **Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(gpus) > 0}')
if gpus:
    print(f'GPU: {gpus[0]}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Install any missing packages
!pip install -q tensorflow keras numpy matplotlib seaborn scikit-learn opencv-python-headless

## Step 2 — Mount Google Drive and Load Dataset

The FER2013 dataset should be uploaded to Google Drive at the following path:
```
MyDrive/fer-emotion-project/dataset/
    train/
        angry/
        disgust/
        fear/
        happy/
        neutral/
        sad/
        surprise/
    test/
        (same structure)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Dataset paths
DRIVE_BASE = '/content/drive/MyDrive/fer-emotion-project'
TRAIN_DIR  = os.path.join(DRIVE_BASE, 'dataset', 'train')
TEST_DIR   = os.path.join(DRIVE_BASE, 'dataset', 'test')

# Output directories (local to Colab session)
MODEL_DIR   = '/content/model'
OUTPUTS_DIR = '/content/outputs'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Verify dataset exists
print('Checking dataset...')
for split, path in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    if os.path.exists(path):
        classes = os.listdir(path)
        total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f'{split}: {len(classes)} classes, {total} images')
    else:
        print(f'ERROR: {split} directory not found at {path}')
        print('Please upload your dataset to Google Drive at the path shown above.')

## Step 3 — Preprocessing and Data Augmentation

### Design Decisions

**Normalization:** Pixel values are scaled from [0, 255] to [0, 1] to improve gradient flow during training.

**Augmentation:** Applied only to training data (not validation or test) to artificially expand the dataset and improve generalization:
- Horizontal flip — faces are symmetric
- Rotation (±10°) — slight head tilts
- Zoom (±10%) — varying camera distances

**Validation split:** 20% of training data is held out for validation. The test set is kept completely untouched until final evaluation to prevent data leakage.

**Class weights:** FER2013 is heavily imbalanced — 'happy' has ~8,000 samples while 'disgust' has only ~547. Class weights penalize the model more for misclassifying rare emotions, forcing it to learn all classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

# Hyperparameters
IMG_SIZE   = 48
BATCH_SIZE = 64
EPOCHS     = 100
NUM_CLASSES = 7

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1,
    validation_split=0.2
)

# Validation generator — rescale only, no augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Test generator — rescale only, shuffle=False for correct label alignment
test_datagen = ImageDataGenerator(rescale=1./255)

print('Loading training data...')
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

print('Loading validation data...')
val_generator = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print('Loading test data...')
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Save class indices for demo.py
import json
class_indices = train_generator.class_indices
with open(os.path.join(OUTPUTS_DIR, 'class_indices.json'), 'w') as f:
    json.dump(class_indices, f)
print(f'\nClass indices: {class_indices}')

# Compute class weights to handle imbalance
print('\nComputing class weights...')
classes = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)
class_weight_dict = dict(zip(np.unique(classes), class_weights_array))
print('Class weights:', class_weight_dict)

In [ ]:
# Visualize class distribution
emotion_labels = list(class_indices.keys())
class_counts = [len(os.listdir(os.path.join(TRAIN_DIR, e))) for e in emotion_labels]

plt.figure(figsize=(10, 5))
bars = plt.bar(emotion_labels, class_counts, color='steelblue', edgecolor='white')
plt.title('Class distribution in training set', fontsize=14)
plt.xlabel('Emotion')
plt.ylabel('Number of images')
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             str(count), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'class_distribution.png'), dpi=150)
plt.show()
print('Saved class_distribution.png')

## Step 4 — Model Architecture (MiniXception)

### Architecture Overview

MiniXception is a lightweight CNN based on the Xception architecture, adapted for small grayscale face images. Key components:

- **Depthwise separable convolutions** — factorize standard convolutions into depthwise + pointwise operations, dramatically reducing parameters while maintaining accuracy
- **Residual connections** — skip connections that help gradients flow during backpropagation, enabling deeper networks
- **Batch normalization** — normalizes activations between layers for faster, more stable training
- **Global average pooling** — replaces fully connected layers, reducing overfitting
- **Softmax output** — produces probability distribution over 7 emotion classes

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation,
    SeparableConv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, add
)
from tensorflow.keras.regularizers import l2

def build_minixception(input_shape=(48, 48, 1), num_classes=7):
    """Build MiniXception model for facial emotion recognition.

    L2 is applied to the stem Conv2D layers and the Dense classifier only: Keras 3
    SeparableConv2D has no kernel_regularizer, and the shipped weights were built this way.
    """
    
    regularization = l2(0.01)
    inputs = Input(input_shape)

    # Initial conv block
    x = Conv2D(8, (3, 3), strides=(1, 1), kernel_regularizer=regularization, use_bias=False, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(8, (3, 3), strides=(1, 1), kernel_regularizer=regularization, use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Xception-style residual modules
    num_filters = [16, 32, 64, 128]
    for num_filter in num_filters:
        residual = Conv2D(num_filter, (1, 1), strides=(2, 2), padding='same', use_bias=False)(x)
        residual = BatchNormalization()(residual)

        x = SeparableConv2D(num_filter, (3, 3), padding='same', use_bias=False)(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = SeparableConv2D(num_filter, (3, 3), padding='same', use_bias=False)(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D((3, 3), strides=(2, 2), padding='same')(x)
        x = add([x, residual])

    # Classification head
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax', kernel_regularizer=regularization)(x)

    return Model(inputs, outputs, name='MiniXception')

model = build_minixception(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES)
model.summary()

## Step 5 — Training

### Training Strategy

- **Optimizer:** Adam with learning rate 0.001 — adaptive learning rate that works well out of the box
- **Loss:** Categorical crossentropy — standard for multi-class classification
- **EarlyStopping:** Stops training if validation loss doesn't improve for 10 epochs, preventing overfitting
- **ReduceLROnPlateau:** Halves the learning rate when validation loss plateaus, squeezing out extra accuracy
- **ModelCheckpoint:** Saves only the best model (by validation accuracy) throughout training

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
import pickle

MODEL_PATH = os.path.join(MODEL_DIR, 'emotion_model.keras')

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

print('Starting training...')
print(f'Model will be saved to: {MODEL_PATH}')
print('-' * 50)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# Save training history
history_path = os.path.join(OUTPUTS_DIR, 'history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(history.history, f)
print(f'\nTraining history saved to {history_path}')

## Step 6 — Evaluation

We evaluate the model on the held-out test set using:
- **Accuracy and loss curves** — show training dynamics and detect overfitting
- **Confusion matrix** — reveals which emotions are confused with each other
- **Classification report** — per-class precision, recall, and F1 score

In [ ]:
# Plot accuracy curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train accuracy', color='steelblue')
ax1.plot(history.history['val_accuracy'], label='Val accuracy', color='orange')
ax1.set_title('Model accuracy over epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train loss', color='steelblue')
ax2.plot(history.history['val_loss'], label='Val loss', color='orange')
ax2.set_title('Model loss over epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('Saved training_curves.png')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions on test set
print('Running predictions on test set...')
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Test accuracy
test_accuracy = np.mean(y_pred == y_true)
print(f'\nTest accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Confusion matrix (normalized)', fontsize=14)
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Saved confusion_matrix.png')

In [ ]:
# Classification report
report = classification_report(y_true, y_pred, target_names=class_labels)
print('Classification Report:')
print(report)

# Save to file
report_path = os.path.join(OUTPUTS_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(f'Test Accuracy: {test_accuracy:.4f}\n\n')
    f.write('Classification Report:\n')
    f.write(report)
print(f'Saved classification_report.txt')

## Step 7 — Download Outputs

Download the trained model and all outputs back to your local machine. You'll need:
- `emotion_model.keras` → copy to your local `model/` folder for `demo.py`
- `class_indices.json` → copy to your local `outputs/` folder for `demo.py`
- All plots → use in your report

In [ ]:
import shutil
from google.colab import files

# Copy everything to Drive for safekeeping
drive_output_dir = os.path.join(DRIVE_BASE, 'outputs')
drive_model_dir  = os.path.join(DRIVE_BASE, 'model')
os.makedirs(drive_output_dir, exist_ok=True)
os.makedirs(drive_model_dir, exist_ok=True)

# Copy outputs to Drive
for f in os.listdir(OUTPUTS_DIR):
    shutil.copy(os.path.join(OUTPUTS_DIR, f), os.path.join(drive_output_dir, f))
shutil.copy(MODEL_PATH, os.path.join(drive_model_dir, 'emotion_model.keras'))

print('All outputs saved to Google Drive!')
print(f'  Model: {drive_model_dir}/emotion_model.keras')
print(f'  Outputs: {drive_output_dir}/')
print()
print('Downloading files to your computer...')

# Download model
files.download(MODEL_PATH)

# Download outputs
for fname in os.listdir(OUTPUTS_DIR):
    files.download(os.path.join(OUTPUTS_DIR, fname))

print('Done! Place emotion_model.keras in your local model/ folder.')
print('Place class_indices.json in your local outputs/ folder.')

## Step 8 — Webcam Demo Note

The live webcam demo (`demo.py`) runs **locally**, not in Colab. After downloading `emotion_model.keras` and `class_indices.json`:

1. Place `emotion_model.keras` in your local `model/` folder
2. Place `class_indices.json` in your local `outputs/` folder  
3. Run locally:
```bash
.venv\Scripts\Activate.ps1
python demo.py
```

The demo will open your webcam, detect faces using OpenCV's Haar cascade, and display the predicted emotion with confidence scores in real time. Press **Q** to quit.

---

## Results Summary

| Metric | Value |
|--------|-------|
| Architecture | MiniXception |
| Dataset | FER2013 (35,887 images) |
| Input size | 48×48 grayscale |
| Number of classes | 7 |
| Expected test accuracy | 60–65% |
| Human-level accuracy on FER2013 | ~65% |

**Note on accuracy:** FER2013 is a notoriously difficult dataset with noisy labels — images collected from Google Image Search often have ambiguous or incorrect emotion labels. A test accuracy of 60–65% is consistent with published results for this architecture and is considered strong performance on this benchmark.